In [ ]:
##PART 3 (SFT-DPO)

In [1]:
!pip install -q transformers accelerate peft trl bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
import os
from google.colab import files
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import DPOConfig, DPOTrainer
from datasets import Dataset
from huggingface_hub import notebook_login

print("Imports loaded!")

Imports loaded!


In [3]:
!pip install -U torchao>=0.16.0

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"\n⏳ Loading {MODEL_NAME}...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

print("✅ Base model loaded!")

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print(" LoRA config ready!")


⏳ Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Base model loaded!
 LoRA config ready!


In [23]:
from trl import DPOConfig, DPOTrainer
from datasets import Dataset
from google.colab import files
import os
import zipfile

print("="*50)
print("PART 3: SFT-DPO WITH TAGGED FORMAT (1000 SAMPLES)")
print("="*50)

# Fix tokenizer warnings
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"
tokenizer.pad_token = tokenizer.eos_token

def format_tagged_dpo(row):
    prompt = f"""Sort these objects from BIGGEST to SMALLEST.

Objects: {row['sample']}

OUTPUT:
1. [ABSOLUTE.LARGEST]:
2. [SECOND.LARGEST]:
3. [MEDIAN.VOLUME]:
4. [SECOND.SMALLEST]:
5. [ABSOLUTE.SMALLEST]: """

    return {
        "prompt": prompt,
        "chosen": row['new_chosen'],
        "rejected": row['reject']
    }

tagged_data = training_data.apply(format_tagged_dpo, axis=1)
tagged_list = tagged_data.tolist()
tagged_dataset = Dataset.from_list(tagged_list)

tagged_dataset = tagged_dataset.select(range(1000))
print(f"Tagged data formatted: {len(tagged_dataset)} samples")

training_args = DPOConfig(
    output_dir="./sft_dpo_output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    save_strategy="no",
    logging_steps=10,
    optim="paged_adamw_8bit",
    report_to="none",
    beta=0.1,
    bf16=False,
    fp16=True,
)

trainer = DPOTrainer(
    model=base_model,
    ref_model=None,
    args=training_args,
    train_dataset=tagged_dataset,
    peft_config=lora_config,
)

print("Starting Part 3 Training (1000 samples, 2 epochs)...")
trainer.train()
print("Part 3 complete!")

trainer.save_model("./sft_dpo_adapter")
print("SFT-DPO adapter saved!")

if os.path.exists("./sft_dpo_adapter"):
    with zipfile.ZipFile("sft_dpo_adapter.zip", 'w') as zipf:
        for root, dirs, files_list in os.walk("./sft_dpo_adapter"):
            for file in files_list:
                zipf.write(os.path.join(root, file),
                          os.path.relpath(os.path.join(root, file), "./sft_dpo_adapter"))
    files.download("sft_dpo_adapter.zip")
    print("sft_dpo_adapter.zip downloaded!")

print("PART 3 COMPLETE!")

PART 3: SFT-DPO WITH TAGGED FORMAT (1000 SAMPLES)
Tagged data formatted: 1000 samples


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Step,Training Loss
10,0.212753
20,0.003442
30,0.000842
40,0.000458
50,0.000069
60,0.000023
70,0.000093
80,0.000766
90,0.000133
100,0.000158


Part 3 complete!
SFT-DPO adapter saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

sft_dpo_adapter.zip downloaded!
PART 3 COMPLETE!


In [25]:
from trl import DPOConfig, DPOTrainer
from datasets import Dataset
from peft import PeftModel
from google.colab import files
import os
import zipfile

print("="*50)
print("PART 3 CONTINUED: Training on remaining samples")
print("="*50)

# Load the current SFT-DPO adapter (trained on first 1000 samples)
model = PeftModel.from_pretrained(base_model, "./sft_dpo_adapter")
print("Loaded existing adapter!")

# Recreate full dataset from training_data
def format_tagged_dpo(row):
    prompt = f"""Sort these objects from BIGGEST to SMALLEST.

Objects: {row['sample']}

OUTPUT:
1. [ABSOLUTE.LARGEST]:
2. [SECOND.LARGEST]:
3. [MEDIAN.VOLUME]:
4. [SECOND.SMALLEST]:
5. [ABSOLUTE.SMALLEST]: """

    return {
        "prompt": prompt,
        "chosen": row['new_chosen'],
        "rejected": row['reject']
    }

tagged_data = training_data.apply(format_tagged_dpo, axis=1)
tagged_list = tagged_data.tolist()
full_dataset = Dataset.from_list(tagged_list)

print(f"Full dataset size: {len(full_dataset)}")

# Get remaining samples (1000 to end)
remaining_dataset = full_dataset.select(range(1000, len(full_dataset)))
print(f"Remaining samples: {len(remaining_dataset)}")

training_args = DPOConfig(
    output_dir="./sft_dpo_output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    save_strategy="no",
    logging_steps=10,
    optim="paged_adamw_8bit",
    report_to="none",
    beta=0.1,
    bf16=False,
    fp16=True,
)

trainer = DPOTrainer(
    model=base_model,
    ref_model=None,
    args=training_args,
    train_dataset=remaining_dataset,
    peft_config=lora_config,
)

print(f"Starting training on remaining samples ({len(remaining_dataset)} samples, 2 epochs)...")
trainer.train()
print("Training complete!")

trainer.save_model("./sft_dpo_adapter")
print("SFT-DPO adapter updated! (All samples trained)")

if os.path.exists("./sft_dpo_adapter"):
    with zipfile.ZipFile("sft_dpo_adapter_full.zip", 'w') as zipf:
        for root, dirs, files_list in os.walk("./sft_dpo_adapter"):
            for file in files_list:
                zipf.write(os.path.join(root, file),
                          os.path.relpath(os.path.join(root, file), "./sft_dpo_adapter"))
    files.download("sft_dpo_adapter_full.zip")
    print("sft_dpo_adapter_full.zip downloaded!")

print("PART 3 FULL COMPLETE!")

PART 3 CONTINUED: Training on remaining samples
Loaded existing adapter!
Full dataset size: 1980
Remaining samples: 980


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/980 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/980 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/980 [00:00<?, ? examples/s]

...

Step,Training Loss
10,0.224621
20,0.003251
30,0.000445
40,0.000147
50,0.001424
60,0.000224
70,0.000088
80,0.000035
90,0.000093
100,0.000077


Training complete!
SFT-DPO adapter updated! (All samples trained)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

sft_dpo_adapter_full.zip downloaded!
PART 3 FULL COMPLETE!


In [26]:
from peft import PeftModel
from transformers import pipeline, GenerationConfig
from transformers import set_seed
import pandas as pd

print("="*50)
print("EVALUATING PART 3 (SFT-DPO FULL)")
print("="*50)

# Load final SFT-DPO adapter
sft_dpo_model = PeftModel.from_pretrained(base_model, "./sft_dpo_adapter")
sft_dpo_model = sft_dpo_model.merge_and_unload()
sft_dpo_model.eval()

generator = pipeline(
    task="text-generation",
    model=sft_dpo_model,
    tokenizer=tokenizer,
    clean_up_tokenization_spaces=False,
)

def generate_response(prompt, generator, seed=42):
    gen_config = GenerationConfig(
        temperature=0.1,
        do_sample=True,
        max_new_tokens=256,
    )
    set_seed(seed)
    messages = [{"role": "user", "content": prompt}]
    outputs = generator(messages, generation_config=gen_config)

    full_output = outputs[0]['generated_text']

    if isinstance(full_output, list):
        for msg in reversed(full_output):
            if msg.get('role') == 'assistant':
                response_text = msg.get('content', '')
                if "OUTPUT:" in response_text:
                    return "OUTPUT: " + response_text.split("OUTPUT:")[-1].strip()
                return response_text.strip()
    if isinstance(full_output, str):
        if "OUTPUT:" in full_output:
            return "OUTPUT: " + full_output.split("OUTPUT:")[-1].strip()
        return full_output.strip()
    return str(full_output)

results = []

for idx, row in testing_data.iterrows():
    prompt = f"""Sort these objects from BIGGEST to SMALLEST.

Objects: {row['sample']}

OUTPUT:
1. [ABSOLUTE.LARGEST]:
2. [SECOND.LARGEST]:
3. [MEDIAN.VOLUME]:
4. [SECOND.SMALLEST]:
5. [ABSOLUTE.SMALLEST]: """

    output = generate_response(prompt, generator)

    results.append({
        'sample_index': idx,
        'sample': row['sample'],
        'llm_output': output
    })

results_df = pd.DataFrame(results)
results_df.to_csv("part3.csv", index=False)

print(" Part 3 results saved to part3.csv")
print("\n First 3 results:")
print(results_df.head(3))
print("\n PART 3 (SFT-DPO FULL) COMPLETE!")

files.download("part3.csv")


EVALUATING PART 3 (SFT-DPO FULL)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

 Part 3 results saved to part3.csv

 First 3 results:
   sample_index                                             sample  \
0             0     paperclip, carbon atom, mouse, toaster, saturn   
1             1        lion, galaxy, coin, skyscraper, carbon atom   
2             2  skyscraper, continent, grain of sand, city, horse   

                                          llm_output  
0  1. [ABSOLUTE.LARGEST]:\n   - paperclip: the la...  
1  1. [ABSOLUTE.LARGEST]: \n   - Lion: the larges...  
2  1. [ABSOLUTE.LARGEST]:\n   - Skyscraper: the t...  

 PART 3 (SFT-DPO FULL) COMPLETE!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
from transformers import pipeline, GenerationConfig
from transformers import set_seed

In [35]:
import pandas as pd
import re

# Load testing data for ground truth
testing_data = pd.read_csv("2015973_testing_data.csv")
gt_lookup = dict(zip(testing_data['sample'], testing_data['ground_truth']))

# Load retest results
df = pd.read_csv("part3_retest.csv")

def parse_tagged_output(output):
    """Extract objects from tagged format"""
    objects = []
    lines = output.split('\n')

    for line in lines:
        # Pattern: 1. [TAG]: object
        match = re.search(r'\d+\.\s*\[.*?\]:\s*(.+)', line)
        if match:
            obj = match.group(1).strip()
            # Clean up extra text
            obj = re.sub(r'^[-*:]\s*', '', obj)
            obj = obj.split('-')[0].strip()
            obj = obj.split(':')[0].strip()
            if obj and obj.lower() not in ['biggest', 'smallest', 'none']:
                objects.append(obj)

    # If no objects found, try to extract from text
    if len(objects) == 0:
        for line in lines:
            # Look for object names in the text
            for obj in ['saturn', 'toaster', 'mouse', 'paperclip', 'carbon atom',
                       'galaxy', 'skyscraper', 'lion', 'coin', 'continent', 'city', 'horse']:
                if obj in line.lower():
                    objects.append(obj)
                    break

    return objects[:5]

def calculate_metrics(df):
    total = len(df)
    fcr_count = 0
    opa_count = 0
    emr_count = 0

    for idx, row in df.iterrows():
        sample = row['sample']
        if sample not in gt_lookup:
            continue

        gt = gt_lookup[sample].split(' -> ')
        llm_objects = parse_tagged_output(row['llm_output'])

        if len(llm_objects) == 5:
            fcr_count += 1
            if llm_objects == gt:
                emr_count += 1

        for i, obj in enumerate(gt):
            if i < len(llm_objects) and llm_objects[i].lower() == obj.lower():
                opa_count += 1

    fcr = fcr_count / total if total > 0 else 0
    opa = opa_count / (5 * total) if total > 0 else 0
    emr = emr_count / total if total > 0 else 0

    return {
        'FCR': round(fcr, 4),
        'OPA': round(opa, 4),
        'EMR': round(emr, 4),
        'FCR_Count': fcr_count,
        'OPA_Count': opa_count,
        'EMR_Count': emr_count,
        'Total': total
    }

# Calculate metrics
metrics = calculate_metrics(df)

print("=" * 50)
print("PART 3 (SFT-DPO) METRICS")
print("=" * 50)
print(f"Total Samples: {metrics['Total']}")
print(f"FCR: {metrics['FCR']:.4f} ({metrics['FCR_Count']}/{metrics['Total']})")
print(f"OPA: {metrics['OPA']:.4f} ({metrics['OPA_Count']}/{(5 * metrics['Total'])})")
print(f"EMR: {metrics['EMR']:.4f} ({metrics['EMR_Count']}/{metrics['Total']})")

PART 3 (SFT-DPO) METRICS
Total Samples: 20
FCR: 0.0000 (0/20)
OPA: 0.0200 (2/100)
EMR: 0.0000 (0/20)
